# Comparison of the B-net variants on the simulated data

This notebook compares the four B-net variants, each trained with two kinds
of input:

- **C**: cooling phase only,
- **H+C**: heating and cooling phase,

on the simulated validation and test sets. The evaluation results of all
trained models are collected in one Excel sheet, which is read here and
summarised in three ways:

1. **Defect detection (IoU)**: range between the worst and the best IoU of
   the predicted defect areas, as a figure and a ranking table.
2. **Depth bias and spread**: median signed depth error and the spread around
   it, per ground-truth depth, as an 8-panel figure and a ranking by distance
   from the ideal point (no bias, no spread).
3. **Depth-estimation error**: MAE / MedAE / RMSE in mm averaged over the
   four depths of each set, as ranking tables.

Figures are written to `FIGURE_DIR` as PDF and PNG.

In [ ]:
import os
import re

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from IPython.display import display

## Settings

In [ ]:
# Excel sheet with the evaluation results and folder for the figures.
RESULTS_FILE = "results/Model_perfomance_small_part_only.xlsx"
FIGURE_DIR = "figures"

# Networks, in the order they are shown, with the short names used in the
# figures and tables and a marker for the scatter plots.
MODEL_ORDER = [
    "Bnet-Mean",
    "Bnet-Vertical projection",
    "Bnet-Deeper regressor",
    "Bnet-End refinement",
]
MODEL_LABELS = {
    "Bnet-Mean": "Mean pooling",
    "Bnet-Vertical projection": "Vertical projection",
    "Bnet-Deeper regressor": "Deeper regressor",
    "Bnet-End refinement": "End refinement",
}
MODEL_MARKERS = {
    "Bnet-Mean": "o",
    "Bnet-Vertical projection": "s",
    "Bnet-Deeper regressor": "^",
    "Bnet-End refinement": "D",
}

# Input modes: colour in the figures, full name in the tables, and the
# background / text colour of the mode cell in the tables.
MODE_ORDER = ["C", "H+C"]
COLORS = {"C": "#4C78A8", "H+C": "#F28E2B"}
MODE_NAMES = {"C": "Cooling only (C)", "H+C": "Heating + cooling (H+C)"}
MODE_CELL_STYLE = {
    "Cooling only (C)": "background-color: #DCEAF5; color: #1F4E79;",
    "Heating + cooling (H+C)": "background-color: #FCE4D6; color: #9C4A09;",
}

DATASET_ORDER = ["Validation", "Test"]

# Excel columns holding the IoU range of each set. The sheet repeats the
# column names for the test set, so pandas adds ".1" to the second
# "Worst IOU"; the validation "Best IOU" header has a leading space.
IOU_COLUMNS = {
    "Validation": {"Best": " Best IOU", "Worst": "Worst IOU"},
    "Test": {"Best": "Best IOU", "Worst": "Worst IOU.1"},
}

# Ground-truth depths [% of thickness] of each set and the suffix pandas adds
# to the repeated per-depth column names (MAE [mm], Median Signed, ...).
# The first four depth blocks in the sheet are the validation set, the next
# four the test set.
DEPTH_GROUPS = {
    "Validation": [(25, ""), (45, ".1"), (65, ".2"), (85, ".3")],
    "Test": [(15, ".4"), (35, ".5"), (55, ".6"), (75, ".7")],
}

os.makedirs(FIGURE_DIR, exist_ok=True)

In [ ]:
def style_ranking_table(table, formats, rank_column, caption=None):
    """Shows a ranking table with coloured input-mode cells and the
    best-ranked row in bold."""
    styled = (
        table.style
        .format(formats)
        .apply(lambda column: [MODE_CELL_STYLE.get(v, "") for v in column], subset=["Mode"])
        .apply(
            lambda row: ["font-weight: bold;" if row[rank_column] == 1 else "" for _ in row],
            axis=1,
        )
        .hide(axis="index")
    )
    if caption is not None:
        styled = styled.set_caption(caption)
    display(styled)


def use_display_names(table):
    """Replaces the model and mode codes by the names used in the paper."""
    table["Model"] = table["Model"].map(MODEL_LABELS).fillna(table["Model"])
    table["Mode"] = table["Mode"].replace(MODE_NAMES)
    return table

## Loading the results table

The first column of the sheet describes each row. It contains two kinds of
labels:

- block headers such as `C no projection` or `H + C cosine projection`,
  which start a group of rows trained with one input mode and one input
  projection,
- model rows such as `Bnet-Mean C`, holding the results of one trained model.

Each model row is assigned the network, input mode and projection it belongs
to. Only the four networks trained without projection are kept.

In [ ]:
results = pd.read_excel(RESULTS_FILE)
label_column = "Unnamed: 0"

# Normalise the labels: strip and collapse repeated spaces.
labels = (
    results[label_column]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

modes, projections, models = [], [], []
current_mode = None
current_projection = None

for label in labels:
    label_lower = label.lower()

    # Block header: sets the mode and projection of the rows that follow.
    block_match = re.fullmatch(
        r"(h\s*\+\s*c|c)\s+(no|cosine|log1p)\s+projection",
        label_lower,
    )
    if block_match:
        current_mode = "H+C" if re.fullmatch(r"h\s*\+\s*c", block_match.group(1)) else "C"
        current_projection = {
            "no": "No projection",
            "cosine": "Cosine projection",
            "log1p": "Log1p projection",
        }[block_match.group(2)]

        modes.append(current_mode)
        projections.append(current_projection)
        models.append(pd.NA)
        continue

    # Model row: network name from the label.
    model = pd.NA
    for name in MODEL_ORDER:
        if name.lower() in label_lower:
            model = name
            break

    # The mode written at the end of the label ("... C", "... H+C") has
    # priority over the mode of the block.
    if re.search(r"h\s*\+\s*c\s*$", label_lower):
        mode = "H+C"
    elif re.search(r"\bc\s*$", label_lower):
        mode = "C"
    else:
        mode = current_mode

    modes.append(mode)
    projections.append(current_projection)
    models.append(model)

results["Analysis mode"] = modes
results["Projection"] = projections
results["Analysis model"] = models

# Keep the four networks, both input modes, without projection, sorted by
# network and then mode (C before H+C).
selected = results.loc[
    results["Analysis model"].isin(MODEL_ORDER)
    & results["Analysis mode"].isin(MODE_ORDER)
    & results["Projection"].eq("No projection")
].copy()

selected["Analysis model"] = pd.Categorical(selected["Analysis model"], categories=MODEL_ORDER, ordered=True)
selected["Analysis mode"] = pd.Categorical(selected["Analysis mode"], categories=MODE_ORDER, ordered=True)
selected = selected.sort_values(["Analysis model", "Analysis mode"]).reset_index(drop=True)

# Check: every network should appear once per mode.
print("Rows retained:", len(selected))
display(
    selected.groupby(["Analysis model", "Analysis mode"], observed=False)
    .size()
    .unstack(fill_value=0)
)
display(selected[[label_column, "Analysis model", "Analysis mode", "Projection"]])

## 1. Defect detection: IoU

For every trained model the sheet gives the worst and the best IoU between
the predicted and the true defect areas over the simulations of each set.
The range between them is drawn as a box (no median is available).

In [ ]:
iou_records = []

for _, row in selected.iterrows():
    for dataset in DATASET_ORDER:
        best = pd.to_numeric(row[IOU_COLUMNS[dataset]["Best"]], errors="coerce")
        worst = pd.to_numeric(row[IOU_COLUMNS[dataset]["Worst"]], errors="coerce")

        if pd.isna(best) or pd.isna(worst):
            continue

        # min / max guard against the two values being swapped in the sheet.
        iou_records.append({
            "Model": row["Analysis model"],
            "Mode": str(row["Analysis mode"]),
            "Dataset": dataset,
            "Worst IoU": min(worst, best),
            "Best IoU": max(worst, best),
        })

iou_df = pd.DataFrame(iou_records)

print("Number of plotted ranges:", len(iou_df))
display(iou_df.head())

In [ ]:
fig, ax = plt.subplots(figsize=(9.0, 5.6))

box_width = 0.18

# Four boxes per network: validation pair on the left, test pair on the
# right; within each pair C on the left, H+C on the right.
position_offsets = {
    ("Validation", "C"): -0.36,
    ("Validation", "H+C"): -0.14,
    ("Test", "C"): 0.14,
    ("Test", "H+C"): 0.36,
}

for model_index, model in enumerate(MODEL_ORDER):
    for dataset in DATASET_ORDER:
        for mode in MODE_ORDER:
            match = iou_df[
                (iou_df["Model"] == model)
                & (iou_df["Dataset"] == dataset)
                & (iou_df["Mode"] == mode)
            ]
            if match.empty:
                continue
            row = match.iloc[0]

            x_position = model_index + position_offsets[(dataset, mode)]

            # Box from the worst to the best IoU; test boxes are hatched.
            ax.add_patch(Rectangle(
                (x_position - box_width / 2, row["Worst IoU"]),
                box_width,
                row["Best IoU"] - row["Worst IoU"],
                facecolor=COLORS[mode],
                edgecolor="black",
                linewidth=1.0,
                hatch=None if dataset == "Validation" else "///",
                alpha=0.92,
                zorder=3,
            ))

ax.set_xlim(-0.65, len(MODEL_ORDER) - 0.35)
ax.set_ylim(0.55, 1.005)
ax.set_ylabel("Intersection over Union (IoU)", fontsize=13)

ax.set_xticks(np.arange(len(MODEL_ORDER)))
ax.set_xticklabels([MODEL_LABELS[m] for m in MODEL_ORDER], rotation=90, fontsize=11)
ax.set_yticks(np.arange(0.55, 1.01, 0.05))
ax.tick_params(axis="y", labelsize=11)
ax.tick_params(axis="x", length=0, pad=7)

ax.yaxis.grid(True, linestyle="--", linewidth=0.7, alpha=0.35, zorder=0)
ax.xaxis.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Thin vertical lines between the networks.
for boundary in np.arange(0.5, len(MODEL_ORDER) - 0.5, 1):
    ax.axvline(boundary, color="#D0D0D0", linewidth=0.7, zorder=0)

legend_handles = [
    Patch(facecolor=COLORS["C"], edgecolor="black", label=MODE_NAMES["C"]),
    Patch(facecolor=COLORS["H+C"], edgecolor="black", label=MODE_NAMES["H+C"]),
    Patch(facecolor="white", edgecolor="black", label="Validation"),
    Patch(facecolor="white", edgecolor="black", hatch="///", label="Test"),
]
ax.legend(handles=legend_handles, loc="lower center", bbox_to_anchor=(0.5, 1.01),
          ncol=2, frameon=False, fontsize=11)

fig.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "iou_no_projection_boxplots.pdf"), bbox_inches="tight")
plt.savefig(os.path.join(FIGURE_DIR, "iou_no_projection_boxplots.png"), dpi=600, bbox_inches="tight")
plt.show()

### IoU ranking

The eight configurations are ranked on the **validation** set only, so the
test set remains an independent check:

1. midpoint of the IoU range, (worst + best) / 2, higher is better
   (a summary of the range, not a mean over simulations),
2. worst IoU (robustness) as the first tie-breaker,
3. best IoU as the second tie-breaker.

The rank the same criteria would give on the test set is shown in the last
column for comparison.

In [ ]:
iou_ranking = iou_df.pivot(index=["Model", "Mode"], columns="Dataset", values=["Worst IoU", "Best IoU"])
iou_ranking.columns = [f"{dataset} {metric}" for metric, dataset in iou_ranking.columns]
iou_ranking = iou_ranking.reset_index()

for dataset in DATASET_ORDER:
    iou_ranking[f"{dataset} midpoint"] = (
        iou_ranking[f"{dataset} Worst IoU"] + iou_ranking[f"{dataset} Best IoU"]
    ) / 2

# Rank on the test set (reported only).
test_order = iou_ranking.sort_values(
    by=["Test midpoint", "Test Worst IoU", "Test Best IoU"],
    ascending=False,
).index
iou_ranking.loc[test_order, "Test rank"] = np.arange(1, len(test_order) + 1)
iou_ranking["Test rank"] = iou_ranking["Test rank"].astype(int)

# Final rank on the validation set.
iou_ranking = iou_ranking.sort_values(
    by=["Validation midpoint", "Validation Worst IoU", "Validation Best IoU"],
    ascending=False,
).reset_index(drop=True)
iou_ranking["Rank"] = np.arange(1, len(iou_ranking) + 1)

iou_ranking = use_display_names(iou_ranking)

iou_columns = [
    "Validation Worst IoU", "Validation Best IoU", "Validation midpoint",
    "Test Worst IoU", "Test Best IoU", "Test midpoint",
]
style_ranking_table(
    iou_ranking[["Rank", "Model", "Mode"] + iou_columns + ["Test rank"]],
    formats={column: "{:.3f}" for column in iou_columns},
    rank_column="Rank",
)

## 2. Depth bias and spread per depth

For every ground-truth depth the sheet gives the **median signed error**
(prediction - ground truth; its sign shows a systematic over- or
underestimation of the depth) and the **standard deviation around that
median** (spread of the predictions). An ideal model sits at (0, 0); the
distance from that point combines bias and spread in one number.

In [ ]:
conditioned_records = []

for _, row in selected.iterrows():
    for dataset, depth_groups in DEPTH_GROUPS.items():
        for depth, suffix in depth_groups:
            median_signed = pd.to_numeric(row[f"Median Signed{suffix}"], errors="coerce")
            std_around_median = pd.to_numeric(row[f"Std around median{suffix}"], errors="coerce")

            if pd.isna(median_signed) or pd.isna(std_around_median):
                continue

            conditioned_records.append({
                "Model": row["Analysis model"],
                "Mode": str(row["Analysis mode"]),
                "Dataset": dataset,
                "Depth [%]": depth,
                "Median signed error": median_signed,
                "Std around median": std_around_median,
                # Euclidean distance from the ideal point (0, 0).
                "Origin distance": np.hypot(median_signed, std_around_median),
            })

conditioned_df = pd.DataFrame(conditioned_records)

print("Depth-conditioned observations:", len(conditioned_df))
display(conditioned_df.head())

In [ ]:
# One panel per set and depth: validation in the top row, test in the bottom.
PANELS = [(dataset, depth) for dataset in DATASET_ORDER for depth, _ in DEPTH_GROUPS[dataset]]
PANEL_LABELS = list("abcdefgh")

# The same axis limits in every panel, so the panels can be compared.
x_limit = conditioned_df["Median signed error"].abs().max() * 1.18
y_limit = conditioned_df["Std around median"].max() * 1.18
maximum_distance = conditioned_df["Origin distance"].max()

# Half circles of equal distance from the ideal point.
contour_radii = np.linspace(maximum_distance / 4, maximum_distance, 4)
theta = np.linspace(0, np.pi, 300)

fig, axes = plt.subplots(2, 4, figsize=(16.0, 8.6), sharex=True, sharey=True)

for ax, (dataset, depth), panel_label in zip(axes.flatten(), PANELS, PANEL_LABELS):
    panel_data = conditioned_df[
        (conditioned_df["Dataset"] == dataset) & (conditioned_df["Depth [%]"] == depth)
    ]

    for radius in contour_radii:
        ax.plot(radius * np.cos(theta), radius * np.sin(theta),
                color="#C8C8C8", linestyle=":", linewidth=0.75, zorder=0)

    # One point per network (marker) and input mode (colour).
    for model in MODEL_ORDER:
        for mode in MODE_ORDER:
            point = panel_data[(panel_data["Model"] == model) & (panel_data["Mode"] == mode)]
            if point.empty:
                continue
            ax.scatter(point["Median signed error"], point["Std around median"],
                       s=65, marker=MODEL_MARKERS[model], facecolor=COLORS[mode],
                       edgecolor="black", linewidth=0.8, zorder=3)

    # Ideal point and zero lines.
    ax.scatter(0, 0, marker="*", s=125, facecolor="black", edgecolor="black", zorder=5)
    ax.axvline(0, color="black", linestyle="--", linewidth=0.75, alpha=0.55, zorder=1)
    ax.axhline(0, color="black", linestyle="--", linewidth=0.75, alpha=0.55, zorder=1)

    ax.set_xlim(-x_limit, x_limit)
    ax.set_ylim(0, y_limit)
    ax.set_title(f"{panel_label}  {dataset}, {depth}%", loc="left", fontsize=11.5, fontweight="bold")
    ax.tick_params(axis="both", labelsize=9.5)
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.supxlabel("Median signed error", fontsize=13, y=0.055)
fig.supylabel("Deviation around median", fontsize=13, x=0.055)

# Legend: colours for the modes, marker shapes for the networks.
mode_handles = [
    Line2D([0], [0], marker="o", linestyle="None", markerfacecolor=COLORS[mode],
           markeredgecolor="black", markersize=8, label=MODE_NAMES[mode])
    for mode in MODE_ORDER
]
model_handles = [
    Line2D([0], [0], marker=MODEL_MARKERS[model], linestyle="None", markerfacecolor="white",
           markeredgecolor="black", markersize=8, label=MODEL_LABELS[model])
    for model in MODEL_ORDER
]
fig.legend(handles=mode_handles + model_handles, loc="upper center",
           bbox_to_anchor=(0.5, 0.995), ncol=6, frameon=False, fontsize=10.5)

fig.tight_layout(rect=[0.06, 0.07, 1.0, 0.90], w_pad=1.6, h_pad=2.0)
plt.savefig(os.path.join(FIGURE_DIR, "depth_conditioned_error_scatter_8_panels.pdf"), bbox_inches="tight")
plt.savefig(os.path.join(FIGURE_DIR, "depth_conditioned_error_scatter_8_panels.png"), dpi=600, bbox_inches="tight")
plt.show()

### Ranking by distance from the ideal point

For every configuration the distance from (0, 0) is averaged over the four
depths of the set (every depth has the same weight). Lower is better; ties
are broken by the mean spread. The mean signed and absolute median errors
and the mean spread are shown for reference.

In [ ]:
def create_distance_ranking(data, dataset):
    ranking = (
        data[data["Dataset"] == dataset]
        .groupby(["Model", "Mode"], observed=True)
        .agg(
            Mean_origin_distance=("Origin distance", "mean"),
            Mean_signed_median=("Median signed error", "mean"),
            Mean_absolute_signed_median=("Median signed error", lambda x: x.abs().mean()),
            Mean_std=("Std around median", "mean"),
        )
        .reset_index()
        .sort_values(["Mean_origin_distance", "Mean_std"])
        .reset_index(drop=True)
    )
    ranking.insert(0, "Rank", np.arange(1, len(ranking) + 1))
    return use_display_names(ranking)


distance_columns = ["Mean_origin_distance", "Mean_signed_median", "Mean_absolute_signed_median", "Mean_std"]

for dataset in DATASET_ORDER:
    style_ranking_table(
        create_distance_ranking(conditioned_df, dataset),
        formats={column: "{:.5f}" for column in distance_columns},
        rank_column="Rank",
        caption=f"{dataset} ranking based on distance from the ideal point",
    )

## 3. Depth-estimation error

MAE, MedAE and RMSE of the predicted depth (in mm) are averaged over the
four depths of each set. The configurations are then ranked on each metric
(1 = lowest error), and two combined scores are formed as the mean of two
metric ranks:

- **MAE + RMSE**: typical error and sensitivity to large errors,
- **MedAE + RMSE**: the same with the median error, which is less affected
  by a few outlying pixels. The tables are sorted by this score.

Ties in a combined score are broken by the lower RMSE, then by the lower
MAE (or MedAE).

In [ ]:
metric_records = []

for _, row in selected.iterrows():
    for dataset, depth_groups in DEPTH_GROUPS.items():
        for depth, suffix in depth_groups:
            mae = pd.to_numeric(row[f"MAE [mm]{suffix}"], errors="coerce")
            medae = pd.to_numeric(row[f"MedAE [mm]{suffix}"], errors="coerce")
            rmse = pd.to_numeric(row[f"RMSE [mm]{suffix}"], errors="coerce")

            if pd.isna(mae) or pd.isna(medae) or pd.isna(rmse):
                continue

            metric_records.append({
                "Dataset": dataset,
                "Depth [%]": depth,
                "Model": row["Analysis model"],
                "Mode": str(row["Analysis mode"]),
                "MAE [mm]": mae,
                "MedAE [mm]": medae,
                "RMSE [mm]": rmse,
            })

metric_df = pd.DataFrame(metric_records)

print("Depth-conditioned metric observations:", len(metric_df))
display(metric_df.head())

In [ ]:
def sequential_rank(table, sort_columns):
    """Unique ranks 1..N after sorting by `sort_columns` (all ascending)."""
    ordered_indices = table.sort_values(by=sort_columns, ascending=True).index
    return table.index.map({index: rank for rank, index in enumerate(ordered_indices, start=1)})


def create_metric_ranking(data, dataset):
    # Mean over the four depths of the set; every depth has the same weight.
    ranking = (
        data[data["Dataset"] == dataset]
        .groupby(["Model", "Mode"], observed=True)
        .agg(
            Mean_MAE=("MAE [mm]", "mean"),
            Mean_MedAE=("MedAE [mm]", "mean"),
            Mean_RMSE=("RMSE [mm]", "mean"),
        )
        .reset_index()
    )

    # Rank on every metric; equal values share the average rank.
    for metric in ["MAE", "MedAE", "RMSE"]:
        ranking[f"{metric} metric rank"] = ranking[f"Mean_{metric}"].rank(method="average", ascending=True)

    # Combined scores: mean of two metric ranks.
    ranking["MAE–RMSE rank score"] = (ranking["MAE metric rank"] + ranking["RMSE metric rank"]) / 2
    ranking["MedAE–RMSE rank score"] = (ranking["MedAE metric rank"] + ranking["RMSE metric rank"]) / 2

    ranking["Rank: MAE + RMSE"] = sequential_rank(ranking, ["MAE–RMSE rank score", "Mean_RMSE", "Mean_MAE"])
    ranking["Rank: MedAE + RMSE"] = sequential_rank(ranking, ["MedAE–RMSE rank score", "Mean_RMSE", "Mean_MedAE"])

    ranking = ranking.sort_values(by="Rank: MedAE + RMSE").reset_index(drop=True)
    return use_display_names(ranking)


for dataset in DATASET_ORDER:
    table = create_metric_ranking(metric_df, dataset)[[
        "Rank: MAE + RMSE", "Rank: MedAE + RMSE", "Model", "Mode",
        "Mean_MAE", "Mean_MedAE", "Mean_RMSE",
        "MAE–RMSE rank score", "MedAE–RMSE rank score",
    ]].rename(columns={
        "Mean_MAE": "Mean MAE [mm]",
        "Mean_MedAE": "Mean MedAE [mm]",
        "Mean_RMSE": "Mean RMSE [mm]",
    })

    style_ranking_table(
        table,
        formats={
            "Mean MAE [mm]": "{:.4f}",
            "Mean MedAE [mm]": "{:.4f}",
            "Mean RMSE [mm]": "{:.4f}",
            "MAE–RMSE rank score": "{:.2f}",
            "MedAE–RMSE rank score": "{:.2f}",
        },
        rank_column="Rank: MedAE + RMSE",
        caption=f"{dataset} ranking based on depth-estimation errors",
    )